# linear-affine-on-custom-tensor — worked example 1: Linear forward builds a matmul-then-add recipe chain

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `linear-affine-on-custom-tensor`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A custom Linear layer computes `out = x @ weight + bias` over your own MiniTensor wrappers. The matmul produces an intermediate that records `weight` and `x` as parents; the bias add produces the output that records the intermediate and `bias` as parents. Each step attaches a Recipe so the reverse pass can later walk the graph.

## Worked solution

We wire the affine map as two recorded steps.

1. **Matmul.** `mm_arr = x.array @ weight.array` gives shape `(B, out_features)`. We wrap it as a MiniTensor `mm` whose `requires_grad` is the OR of its inputs', then attach a Recipe naming the matmul func with parents `{0: x, 1: weight}`.
2. **Bias add.** `out_arr = mm.array + bias.array`; the bias of shape `(out_features,)` broadcasts across the batch axis. We wrap as `out` and attach a second Recipe with parents `{0: mm, 1: bias}`.
3. **Why two Recipes.** In a full framework, `wrap_forward_fn` would build these automatically when you call wrapped ops. Constructing them by hand keeps the focus on the affine math: matmul then broadcast-add.

The demo runs a `(B=2, in=3)` input through `(in=3, out=4)` weights and a `(4,)` bias, prints the output shape, and confirms the recipe parents are recorded.

In [ ]:
import numpy as np
from dataclasses import dataclass

np.random.seed(0)

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def linear_forward(x, weight, bias):
    mm_arr = x.array @ weight.array
    mm = MiniTensor(mm_arr, requires_grad=(x.requires_grad or weight.requires_grad))
    mm.recipe = Recipe(np.matmul, (x.array, weight.array), {}, {0: x, 1: weight})
    out_arr = mm.array + bias.array
    out = MiniTensor(out_arr, requires_grad=(mm.requires_grad or bias.requires_grad))
    out.recipe = Recipe(np.add, (mm.array, bias.array), {}, {0: mm, 1: bias})
    return out

x = MiniTensor(np.random.randn(2, 3))
weight = MiniTensor(np.random.randn(3, 4), requires_grad=True)
bias = MiniTensor(np.random.randn(4), requires_grad=True)
out = linear_forward(x, weight, bias)
print('out shape:', out.array.shape, '| requires_grad:', out.requires_grad)
print('recipe parents:', sorted(out.recipe.parents.keys()))